# 🟡 Solution: MaxPool2D Forward & Backward (NumPy)

In [ ]:
import numpy as np

In [ ]:
# ✅ SOLUTION

def maxpool2d(x, kernel_size=2, stride=2):
    # x: (N, C, H, W)
    N, C, H, W = x.shape
    k, s = kernel_size, stride

    H_out = (H - k) // s + 1
    W_out = (W - k) // s + 1

    out = np.zeros((N, C, H_out, W_out), dtype=np.float64)
    argmax = np.zeros((N, C, H_out, W_out), dtype=np.int64)   # flat index inside each window

    for i in range(H_out):
        for j in range(W_out):
            window = x[:, :, i * s:i * s + k, j * s:j * s + k].reshape(N, C, -1)
            idx = window.argmax(axis=2)
            argmax[:, :, i, j] = idx
            out[:, :, i, j] = np.take_along_axis(window, idx[:, :, None], axis=2)[:, :, 0]

    n_idx, c_idx = np.indices((N, C))

    def backward(dout):
        # Pure routing: each upstream value lands on its window's winner, everything else stays 0
        dx = np.zeros((N, C, H, W), dtype=np.float64)
        for i in range(H_out):
            for j in range(W_out):
                di, dj = np.divmod(argmax[:, :, i, j], k)      # unflatten the window offset
                np.add.at(dx, (n_idx, c_idx, i * s + di, j * s + dj), dout[:, :, i, j])
        return dx

    return out, backward

In [ ]:
# Verify
np.random.seed(0)
x = np.random.randn(2, 3, 8, 8)
out, backward = maxpool2d(x, kernel_size=2, stride=2)
print("out shape:", out.shape)
print("dx  shape:", backward(np.ones_like(out)).shape)

tiny = np.array([[[[1.0, 2.0], [3.0, 9.0]]]])
o, bwd = maxpool2d(tiny, 2, 2)
print("max      :", o.ravel())
print("routed dx:", bwd(np.array([[[[5.0]]]])).ravel(), "(expect [0 0 0 5])")

In [ ]:
from torch_judge import check
check("numpy_maxpool2d")